# 保存モデルの評価と可視化

学習済みモデルを読み込み，オペランドの桁数ごとに完全一致率を求める．
評価用コードを使い，ヒートマップと折れ線グラフを表示する．

- [configs/addition/evaluate.yaml](../configs/addition/evaluate.yaml)：評価条件
- [scripts/evaluate.sh](../scripts/evaluate.sh)：評価の実行コマンド
- [src/evaluate.py](../src/evaluate.py)：評価の入口
- [support/length_evaluation.py](../support/length_evaluation.py)：評価・集計・描画の実装

[CALT入門](calt-tutorial.ipynb)でモデルを学習・保存してから進める．

## 準備

ColabのGPU・ランタイム`2026.07`を使用する．Google Driveを接続し，教材フォルダへ移動する．

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/project-research

In [ ]:
%%capture
%pip install calt-x matplotlib

## 1. 評価条件の確認

[configs/addition/evaluate.yaml](../configs/addition/evaluate.yaml)を開き，評価するモデルに合わせて設定する．

| 設定 | 内容 |
|---|---|
| `model_path` | 保存したモデルとtokenizerのフォルダ |
| `format_function` | 学習時の表記関数．`モジュール名:関数名`で指定 |
| `answer_function` | 正解を計算する関数．加算は`operator:add`，乗算は`operator:mul` |
| `lexer_config` | 学習時のlexer設定ファイル |
| `digits` | 評価する桁数．指定した桁数の全組合せで評価 |
| `samples_per_cell` | 桁数の組ごとの問題数 |
| `seed` | 評価問題を生成する乱数シード |
| `batch_size` | 生成時にまとめて処理する問題数 |
| `max_length` | 答えの生成上限．終了tokenを含むtoken数 |
| `output_dir` | CSVと図の保存先 |

例えば`digits: [1, 2, 3, 4, 5]`なら，1桁＋1桁から5桁＋5桁まで25通りの桁数の組を評価する．各組では異なる数値ペアを抽出する．
設定を保存してから，次のセルで内容を確認する．

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/addition/evaluate.yaml')
print(OmegaConf.to_yaml(cfg))

### 学習時と同じ表記を使う

`format_function: src.addition:format_problem`は，`src/addition.py`の`format_problem`を使う指定である．
学習後にこの関数を変更した場合は，学習時の関数を別ファイルに保存して指定する．例えば`src/format_experiment1.py`に保存した関数なら，`src.format_experiment1:format_problem`とする．lexerと独自の埋め込みも学習時の実装・設定を使う．

指定した関数に問題と正解を渡し，表記を確認する．

In [ ]:
from support.length_evaluation import load_function

format_problem = load_function(cfg.format_function)
calculate = load_function(cfg.answer_function)
a, b = 12, 345
format_problem([a, b], calculate(a, b))

## 2. 評価の実行

設定に従って評価問題を生成し，保存モデルで答えを生成する．学習は行わない．
正解を途中まで与えず，答えを最初から生成する評価である．
表記変換後の正解と，生成した答えを終了tokenまで比較し，完全一致を判定する．

In [ ]:
!bash scripts/evaluate.sh

## 3. 桁数別のヒートマップ

行は第1オペランドの桁数`m`，列は第2オペランドの桁数`n`，色はその組の完全一致率を表す．
桁数は0埋め・逆順などの変換前の整数から求める．

In [ ]:
from IPython.display import Image, display

display(Image(filename=f'{cfg.output_dir}/length_heatmap.png'))

## 4. 同じ桁数同士の完全一致率

横軸は各オペランドの桁数`n`，縦軸は`n`桁同士の問題の完全一致率である．ヒートマップの対角成分を取り出して描く．

In [ ]:
display(Image(filename=f'{cfg.output_dir}/length_curve.png'))

## 5. 個々の生成結果を確認する

`predictions.csv`には，元の数値・桁数・モデルへの入力・正解・生成結果・終了tokenの有無・完全一致の判定を保存している．
`exact`が`True`なら完全一致，`False`なら不一致である．

In [ ]:
import csv

with open(f'{cfg.output_dir}/predictions.csv') as f:
    results = list(csv.DictReader(f))
results[:5]

不一致になった例を表示する．

In [ ]:
[row for row in results if row['exact'] == 'False'][:5]

## 6. CSVから図を描き直す

保存した`predictions.csv`から再集計・描画する．モデルの再読み込みや生成は行わない．
集計は`accuracy_by_digits.csv`，同じ桁数同士の集計は`accuracy_same_digits.csv`に保存される．

In [ ]:
from support.length_evaluation import plot_results

summary = plot_results(f'{cfg.output_dir}/predictions.csv')
summary[:5]

In [ ]:
display(Image(filename=f'{cfg.output_dir}/length_heatmap.png'))
display(Image(filename=f'{cfg.output_dir}/length_curve.png'))